# TorGen: DETR-CVAE for Tornado Outbreak Generation

Install the package, mount Drive, and train.

In [ ]:
!pip install -q git+https://github.com/jcaramichaellehigh/TorGen.git

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Optional: experiment tracking
# Sign up at wandb.ai, get API key from wandb.ai/authorize
try:
    import wandb
    wandb.login()
except ImportError:
    print("wandb not installed, skipping experiment tracking")

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

from torgen.training import TrainConfig, train

# --- First: 5-epoch sanity check ---
config = TrainConfig(
    drive_dir="/content/drive/MyDrive/raw",
    checkpoint_dir="/content/drive/MyDrive/checkpoints",
    max_epochs=5,
    patience=999,  # don't early-stop during sanity check
)

trainer = train(config)

# Check that loss is decreasing
losses = trainer.train_losses
print("\n--- Sanity Check ---")
print(f"Epoch 1 loss: {losses[0]:.4f}")
print(f"Epoch 5 loss: {losses[-1]:.4f}")
print(f"Decreasing: {losses[-1] < losses[0]}")
if losses[-1] >= losses[0]:
    print("WARNING: Loss not decreasing. Check learning rate or data.")
else:
    print("Looks healthy. Run the next cell for the full training run.")

In [ ]:
# --- Full training run ---
# Delete the 5-epoch checkpoint so we start fresh
import os
for f in ["checkpoint_latest.pt", "checkpoint_best.pt"]:
    path = os.path.join("/content/drive/MyDrive/checkpoints", f)
    if os.path.exists(path):
        os.remove(path)

config = TrainConfig(
    drive_dir="/content/drive/MyDrive/raw",
    checkpoint_dir="/content/drive/MyDrive/checkpoints",
)

trainer = train(config)

In [ ]:
print(f"Epochs trained: {trainer.epoch}")
print(f"Final train loss: {trainer.train_losses[-1]:.4f}")
print(f"Final val loss: {trainer.val_losses[-1]:.4f}")
print(f"Best val loss: {trainer.best_val_loss:.4f}")